In [1]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/CFAT_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/CFAT_M2L8


/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
# import rasterio
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-01-25 13:11:13.420207: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use only GPU 1

In [4]:
from model import cfat
model = cfat.CFAT(in_chans=7, upscale=16).to('cuda')

/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torchvision/transforms/functional_tensor.py:5: UserWarning: The torchvision.transforms.functional_tensor module is deprecated in 0.15 and will be **removed in 0.17**. Please don't rely on it. You probably just need to use APIs in torchvision.transforms.functional or in torchvision.transforms.v2.functional.
  warnings.warn(
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [5]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer=optimizer, milestones=[225, 350, 400, 450], gamma=0.5)

In [6]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

# filenames = ['/content/drive/MyDrive/GeoSR/L8MODIS_30000_2/L8MODIS.tfrecords']
# ds = input_pipeline(filenames, batch_size=5, is_shuffle=False, is_train=True, is_repeat=True)


# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))*0.0001
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))*0.0000275-0.2

#         axes[0].imshow(lres_img[:, :, [0, 3, 2]]*2)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, 3:0:-1]*2)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break


In [ ]:
def load_matched_weights(model, checkpoint_path):
    # Load checkpoint (state_dict)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint

    # Filter only matching keys
    model_dict = model.state_dict()
    matched_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

    # Load the matched weights
    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Loaded {len(matched_dict)} matching parameters out of {len(model_dict)} total.")

    return model


model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/CFAT_x4_epoch_weights_M2L8_new.pth')


# SR Training

In [ ]:
total_epochs = 30

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']

ds = input_pipeline(filenames, batch_size=8, is_shuffle=True, is_train=True, is_repeat=False)

for epoch in range(total_epochs):
    print(f'Epoch {epoch}')

    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
    
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)


        optimizer.zero_grad()
        output = model(lr)

        l_total = 0

        # pixel loss
        l_pix = torch.nn.L1Loss()(output, hr)
        l_total += l_pix

        l_total.backward()
        optimizer.step()

        if step % 500 == 0:
            print(f'Step {step}, Loss: {l_total.item()}')

    torch.save(model.state_dict(), '/glade/derecho/scratch/lizhili/m2l8/sr_ds/CFAT_x16_epoch_weights_M2L8_new.pth')


# Downstream SR Finetuning

In [9]:

def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):

    # optimizer = optim.Adam(model.parameters(), lr=2e-4)
    model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_epoch_weights_M2L8_new.pth')
    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    #--------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 2, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
            lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

            optimizer.zero_grad()
            output = model(lr)

            l_total = 0

            # pixel loss
            l_pix = torch.nn.L1Loss()(output, hr)
            l_total += l_pix

            l_total.backward()
            optimizer.step()

            if step % 500 == 0:
                print(f'Step {step}, Loss: {l_total.item()}')

        torch.save(model.state_dict(), finetuned_model)
    #--------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_sample, is_shuffle=False, is_train=True, is_repeat=False)
    for step, (lr, _, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)

        with torch.no_grad():
            output = model(lr)
        output = output.detach().cpu().numpy()
        # output = np.clip(output, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        output[output<0]=0
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(2*lr_show[:, :, [0,3,2]])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(2*(output_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  
label_size = 1000 
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_CFAT_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  
label_size = 1000 
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_CFAT_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  
label_size = 1000 
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_CFAT_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  
label_size = 1000 
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_CFAT_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  
label_size = 1000 
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/CFAT_x16_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_CFAT_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)